In [1]:
!pip install -q scanpy
!pip install -q scvi-tools
!pip install -q leidenalg
!pip install -q diffxpy;
!pip install -q soupx-python
!pip install -q celltypist
!pip install -q decontx-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import anndata as ad
import leidenalg
import os
import diffxpy.api as de;
import scipy.sparse as sp
import scvi
import soupx;
import celltypist
from celltypist import models
import decontx

/usr/local/lib/python3.12/dist-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [6]:
import subprocess
files_dir = '/content/COVID_PBMC_1'

links = ['https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557327/suppl/GSM4557327%5F555%5F1%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557328/suppl/GSM4557328%5F555%5F2%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557329/suppl/GSM4557329%5F556%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557330/suppl/GSM4557330%5F557%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557331/suppl/GSM4557331%5F558%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557332/suppl/GSM4557332%5F559%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557333/suppl/GSM4557333%5F561%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557334/suppl/GSM4557334%5FHIP002%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557335/suppl/GSM4557335%5FHIP015%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557336/suppl/GSM4557336%5FHIP023%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557337/suppl/GSM4557337%5FHIP043%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557338/suppl/GSM4557338%5FHIP044%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557339/suppl/GSM4557339%5FHIP045%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz']
for link in links:
  subprocess.run(['wget', '-P', files_dir, link], check = True)# suprocess is a way to type into the terminal but doing it from python,
  subprocess.run(f'gunzip {files_dir}/*.gz', shell = True, check = True)
  # this line is = to typing this in bash wget -P /content/my_downloads https://example.com/file.rds.gz
  #the -P flag means direct my downloads to this folder.


In [14]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects import r
from pathlib import Path

folder_path = Path('/content/COVID_PBMC_1')
pandas2ri.activate()
readRDS = ro.r['readRDS']
adata_dict = {}

donor_dict = {"GSM4557330": "covid_557",
              "GSM4557339": "HIP045",
              "GSM4557327": "covid_555_1",
              "GSM4557331": "covid_558",
              "GSM4557336": "HIP023",
              "GSM4557335": "HIP015",
              "GSM4557329": "covid_557",
              "GSM4557328": "covid_555_2",
              "GSM4557338": "HIP044",
              "GSM4557334": "HIP002",
              "GSM4557337": "HIP043",
              "GSM4557333": "covid_561",
              "GSM4557332": "covid_559",
               }
for item in folder_path.iterdir():#create a loop to traverse through the folder and create adata objects
  obj = readRDS(str(item.resolve())) #returns the path of the file as a string which readrds expects
  exon = obj.rx2("exon")#the rds file contains 3 matrices exon, introns, and spanning, exon is the standard gene-expression matrix

  # convert R sparse matrix → SciPy sparse
  i = np.array(exon.slots["i"])
  p = np.array(exon.slots["p"])
  x = np.array(exon.slots["x"])
  dims = tuple(exon.slots["Dim"])

  X = sp.csc_matrix((x, i, p), shape=dims).T

  # build AnnData
  adata = ad.AnnData(X)
  adata.var_names = list(r['rownames'](exon))
  adata.obs_names = list(r['colnames'](exon))

  #extracts unique sample id and assigns it to a col
  sample_id = item.name.split('_')[0]
  adata.obs['sample'] = sample_id

  #extracts condition of the sample and assigns it to a col
  healthy_sample = item.name.split('_')[1]
  if healthy_sample.startswith('HIP'):
    adata.obs['condition'] = 'HEALTHY'
  else:
    adata.obs['condition'] = 'COVID'

  adata.obs['donor'] = adata.obs['sample'].map(donor_dict)

  adata_dict[sample_id] = adata

In [15]:
for sample_id, adata in adata_dict.items():
  print(sample_id, adata.shape)

GSM4557338 (11740, 31540)
GSM4557336 (8343, 30937)
GSM4557329 (7465, 31401)
GSM4557327 (17057, 34463)
GSM4557332 (9665, 30484)
GSM4557337 (12578, 33635)
GSM4557328 (9663, 30469)
GSM4557339 (13883, 33098)
GSM4557331 (21215, 33454)
GSM4557335 (5330, 30098)
GSM4557334 (9070, 32600)
GSM4557333 (7825, 32063)
GSM4557330 (16536, 33852)


In [19]:
#since i do not have the raw counts matrices for this data i cannot
#use soupx for this ambient rna removal so instead i will be test
#decontx. i will also save the processed adata as h5ad files

#output_dir = Path('decontx_files_h5ad')
#output_dir.mkdir(exist_ok=True)
#adata.write_h5ad(output_dir / f'{sample_id}_decontx.h5ad')
for sample_id, adata in adata_dict.items():
  adata.layers['raw_data'] = adata.X
  temp = adata.copy()
  sc.pp.filter_cells(temp, min_genes = 200)
  sc.pp.filter_genes(temp, min_cells = 10)
  sc.pp.normalize_total(temp, target_sum = None, inplace = True)# inplace  = true means modify the adata directly instead of returning a new adata object
  sc.pp.log1p(temp)
  sc.pp.pca(temp)
  sc.pp.neighbors(temp, n_neighbors = 30)
  sc.tl.leiden(temp, key_added = 'decontx_clusters', flavor = 'igraph', n_iterations = 2, directed = False)
  decontx.decontx(temp, cluster_key = 'decontx_clusters')
  adata.layers['decontX_counts'] = temp.layers['decontX_counts'].copy()
  adata.X = temp.layers['decontX_counts'].copy()
  del temp



Starting DecontX
Processing 8903 cells, 15548 genes
Using 10 clusters from 'decontx_clusters'
Processing as single batch
Iter 0: LL=-35032874.7, change=0.2641, mean_contam=0.468
Iter 10: LL=-34937860.6, change=0.0281, mean_contam=0.322
Iter 20: LL=-34921729.9, change=0.0116, mean_contam=0.258
Iter 30: LL=-34913512.3, change=0.0073, mean_contam=0.215
Iter 40: LL=-34908684.3, change=0.0052, mean_contam=0.184
Iter 50: LL=-34905564.6, change=0.0045, mean_contam=0.160
Iter 60: LL=-34903544.5, change=0.0040, mean_contam=0.142
Iter 70: LL=-34902234.7, change=0.0037, mean_contam=0.128
Iter 80: LL=-34901442.0, change=0.0031, mean_contam=0.117
Iter 90: LL=-34901048.4, change=0.0032, mean_contam=0.108
Iter 100: LL=-34900737.8, change=0.0032, mean_contam=0.101
Iter 110: LL=-34900529.6, change=0.0032, mean_contam=0.095
Iter 120: LL=-34900483.9, change=0.0027, mean_contam=0.090
Iter 130: LL=-34900474.5, change=0.0026, mean_contam=0.086
Iter 140: LL=-34900535.6, change=0.0030, mean_contam=0.083
Iter 

ValueError: Value passed for key 'decontX_counts' is of incorrect shape. Values of layers must match dimensions ('obs', 'var') of parent. Value had shape (8903, 15548) while it should have had (11740, 31540).

In [ ]:
for sample_id, adata in adata_dict.items():
  print(f'before light qc filter: {adata.X.shape}')
  sc.pp.filter_cells(adata, min_genes = 200)
  sc.pp.filter_genes(adata, min_cells = 10)
  print(f'after light qc filter: {adata.X.shape}')
  adata.var['mt'] = adata.var.index.str.startswith('MT-')
  adata.var['rb'] = adata.var.index.str.startswith(("RPS", "RPL","FAU", "MRPL", "RSL"))
  adata.var["hb"] = adata.var.index.str.startswith(r"^HB[ABDEGMQZ]\d*(?!\w)")
  sc.pp.calculate_qc_metrics(adat, qc_vars = ['mt', 'rb', 'hb'], percent_top = [20], log1p = True, inplace = True)

  p1 = sns.displot(adata.obs['total_counts'], bin = 100, kde = False)
  p2 = sc.pl.violin(adata, 'pct_counts_mt')
  p3 = sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color = 'pct_counts_mt')
